# Índice

- **1. Setup, dados e protocolo** (Ambiente Virtual e Dependências)
- **2. Modelo VAE base**
- **3. Melhorias manuais no VAE**
    -  Comparativo Técnico: VAE Base vs. Improved VAE
- **4. Otimização de Hiperparâmetros: Grid Search (Subset 20%)**

- **5. Treino final no conjunto completo com Tracking de Métricas**
- **6. Estudo Comparativo: Impacto das Funções de Perda na Qualidade Generativa**
    - 6.2 Perceptual Loss (VGG-16)
    - 6.3 SSIM Loss (Structural Similarity Index)
    - 6.4 Super Loss (Abordagem Híbrida)
- **Extra: Inadequação do Denoising Autoencoder (DAE) como Modelo Generativo**
- **EXTRA: Denoising Autoencoder (DAE) - Demonstração de falha gerativa**
- **Notas Finais: Análise Cruzada de Autoencoders (VAE vs. DAE)**


## Setup do Ambiente Virtual e Dependências

Para garantir que todas as dependências estão isoladas, recomenda-se criar um ambiente virtual (`venv`) antes de correr o notebook. Abre o terminal na pasta do projeto e executa:

```bash
#python3 -m venv .venv
#source .venv/bin/activate
#pip install --upgrade pip
pip install torch torchvision matplotlib datasets pillow "torchmetrics[image]" tqdm
pip install ipywidgets
```

*Nota: No VSCode, após criares o `.venv`, clica no canto superior direito para escolheres o kernel deste novo ambiente.*

## 1. Setup, dados e protocolo

O workflow segue a regra do novo trabalho:
1. `dev_loader` e `dev_loader_aug` para desenho/afinacao;
2. `full_train_loader` e `full_train_loader_aug` apenas para o treino final;
3. avaliacao final com `5.000` amostras e `10` repeticoes.

In [ ]:
import sys
import torch
print(f"Versão do Torch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"Caminho do Python: {sys.executable}")


In [ ]:
# ==========================================
# SETUP CONSOLIDADO E COMPLETO (VAE)
# ==========================================
from __future__ import annotations
import sys
import random
import csv
import json
import inspect
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Importar métricas de avaliação (FID e KID)
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
except ImportError:
    print("AVISO: torchmetrics[image] não encontrado. Instala com: pip install torchmetrics[image]")
    FrechetInceptionDistance = None
    KernelInceptionDistance = None

# 1. Configurações de Reprodução e Caminhos
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Ajuste automático do caminho conforme a pasta
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'VAE', 'Diffusion_model', 'GAN', 'DAE'}:
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
KAGGLE_ROOT = PROJECT_ROOT / 'ArtBench-10'
TRAINING_CSV_PATH = PROJECT_ROOT / 'student_start_pack' / 'training_20_percent.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Importar helper de carregamento de dados
from artbench_local_dataset import load_kaggle_artbench10_splits

# 2. Definições de Dispositivo e Constantes
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 2

def safe_num_workers(requested: int) -> int:
    if "ipykernel" in sys.modules and int(requested) > 0:
        return 0
    return int(requested)

# 3. Transforms (BASE e AUGMENTED)
BASE_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
])

AUGMENTED_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
])

# 4. Dataset e Funções de Utilidade
class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform else img
        return x, y, real_idx

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Ficheiro não encontrado: {csv_path}")
    ids = []
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            v = str(row.get(index_column, "")).strip()
            if v: ids.append(int(v))
    return ids

def build_loader(indices, transform, shuffle=True, batch_size=BATCH_SIZE):
    ds = HFDatasetTorch(train_hf, transform=transform, indices=indices)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )

def plot_loss_curves(history, title): 
    plt.figure(figsize=(8, 4))
    for key, values in history.items():
        if values and isinstance(values[0], (int, float)):
            plt.plot(values, label=key)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 5. Carregar Dados Reais e Criar TODOS os Loaders
print(f"Lendo ArtBench-10 de: {KAGGLE_ROOT}...")
try:
    hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
    train_hf = hf_ds["train"]
    test_hf = hf_ds["test"]
    class_names = list(train_hf.features['label'].names)

    # Identificadores para Subset (20%) e Completo (100%)
    subset_ids = load_ids_from_training_csv(TRAINING_CSV_PATH)
    full_train_ids = list(range(len(train_hf)))

    # Loaders de Desenvolvimento (Subset 20%)
    dev_loader = build_loader(subset_ids, BASE_TRANSFORM, shuffle=True)
    dev_loader_aug = build_loader(subset_ids, AUGMENTED_TRANSFORM, shuffle=True)
    
    # Loaders de Treino Completo (100%)
    full_train_loader = build_loader(full_train_ids, BASE_TRANSFORM, shuffle=True)
    full_train_loader_aug = build_loader(full_train_ids, AUGMENTED_TRANSFORM, shuffle=True)
    
    # Loader de Teste
    test_loader = DataLoader(
        HFDatasetTorch(test_hf, transform=BASE_TRANSFORM),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )
    print('--- Setup Concluído com Sucesso ---')
except Exception as e:
    print(f"Erro ao carregar dados: {e}")

print('Dispositivo:', device)

## 2. Modelo VAE base

In [ ]:
## 2. Modelo VAE base

import numpy as np
import json
import inspect

# Métricas: CUDA suporta float64, MPS não — forçar CPU em MPS (CORRIGIDO)
if torch.backends.mps.is_available():
    METRICS_DEVICE = torch.device('cpu') 
else:
    METRICS_DEVICE = device

class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128, base_channels=128, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        hidden_dim = base_channels * 4 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss(recon, x, mu, logvar, beta=0.5):
    recon_loss = F.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_loss + beta * kl
    return total, recon_loss.detach(), kl.detach()


@torch.no_grad()
def sample_vae(model, n_samples, seed=None):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)


@torch.no_grad()
def show_vae_reconstructions(model, loader, max_images=8):
    x, _, _ = next(iter(loader))
    x = x[:max_images].to(device)
    recon, _, _ = model(x)
    grid = torch.cat([x.cpu(), recon.cpu()], dim=0)
    plt.figure(figsize=(12, 4))
    plt.imshow(np.clip(make_grid(grid, nrow=max_images).permute(1, 2, 0).numpy(), 0, 1))
    plt.axis('off')
    plt.title('Linha 1: original | Linha 2: reconstruida')
    plt.show()


def train_vae(model, train_loader, *, epochs=20, lr=2e-4, beta=0.5, run_name='vae_run'):
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'recon_loss': [], 'kl_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = total_recon = total_kl = 0.0
        progress = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False)
        for x, _, _ in progress:
            x = x.to(device)
            optimizer.zero_grad(set_to_none=True)
            recon, mu, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, beta=beta)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
            total_recon += float(recon_loss.item())
            total_kl += float(kl_loss.item())
            progress.set_postfix(loss=f'{loss.item():.4f}')

        n_batches = max(1, len(train_loader))
        history['train_loss'].append(total_loss / n_batches)
        history['recon_loss'].append(total_recon / n_batches)
        history['kl_loss'].append(total_kl / n_batches)
        print(f'Epoch {epoch:03d}/{epochs} | loss={history["train_loss"][-1]:.4f} | recon={history["recon_loss"][-1]:.4f} | kl={history["kl_loss"][-1]:.4f}')

    torch.save(model.state_dict(), run_dir / 'last_model.pt')
    with open(run_dir / 'history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)
    return history, run_dir

# -----------------------------------------------------------------------------------------
# AVALIAÇÃO — protocolo do enunciado (FID + KID, 10 repetições)
# -----------------------------------------------------------------------------------------

N_EVAL_SAMPLES  = 5000
N_EVAL_REPEATS  = 10
KID_SUBSETS     = 50
KID_SUBSET_SIZE = 100
EVAL_BATCH_SIZE = 128 # Lotear o cálculo das métricas para evitar CUDA Out of Memory

def collect_real_images(loader, n):
    """Recolhe n imagens reais de uma vez. Retorna float32 [0,1] em CPU."""
    collected, seen = [], 0
    for x, _, _ in loader:
        take = min(x.size(0), n - seen)
        collected.append(x[:take].cpu().float())
        seen += take
        if seen >= n:
            break
    return torch.cat(collected, dim=0)[:n]

@torch.no_grad()
def sample_vae_batched(model, n_samples, seed=None, batch_size=EVAL_BATCH_SIZE):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    
    out = []
    for i in range(0, n_samples, batch_size):
        out.append(model.decode(z[i:i+batch_size]).clamp(0.0, 1.0).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def evaluate_vae(model, real_images, repeats=N_EVAL_REPEATS, base_seed=0):
    fid_scores, kid_means, kid_stds = [], [], []

    for rep in range(repeats):
        seed = base_seed + rep
        print(f'  Rep {rep+1:02d}/{repeats}  (seed={seed})', end='  ', flush=True)

        fake_images = sample_vae_batched(model, N_EVAL_SAMPLES, seed=seed)

        # normalize=False para usarmos [0, 255] que é mais estável
        fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
        kid_metric = KernelInceptionDistance(
            feature=2048, subsets=KID_SUBSETS, subset_size=KID_SUBSET_SIZE, normalize=False
        ).to(METRICS_DEVICE)

        for i in range(0, N_EVAL_SAMPLES, EVAL_BATCH_SIZE):
            # Conversão para byte antes de passar às métricas
            real_batch = (real_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fake_batch = (fake_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            
            fid_metric.update(real_batch, real=True)
            fid_metric.update(fake_batch, real=False)
            
            kid_metric.update(real_batch, real=True)
            kid_metric.update(fake_batch, real=False)

        fid_val = float(fid_metric.compute().item())
        kid_mean, kid_std = kid_metric.compute()
        
        
        fid_scores.append(fid_val)
        
        kid_means.append(float(kid_mean.item()))
        kid_stds.append(float(kid_std.item()))

        print(f'FID = {fid_val:.4f} | KID = {kid_means[-1]:.6f} ± {kid_stds[-1]:.6f}')

        fid_metric.reset()
        kid_metric.reset()

    results = {
        'n_samples':        N_EVAL_SAMPLES,
        'repeats':          repeats,
        'fid_per_rep':      fid_scores,
        'kid_mean_per_rep': kid_means,
        'kid_std_per_rep':  kid_stds,
        'fid_mean':         float(np.mean(fid_scores)),
        'fid_std':          float(np.std(fid_scores)),
        'kid_mean':         float(np.mean(kid_means)),
        'kid_std':          float(np.std(kid_means)),
    }

    print(f"\n{'='*50}")
    print(f"RESULTADOS FINAIS — AVALIAÇÃO ({repeats} REPETIÇÕES)")
    print(f"{'='*50}")
    print(f"FID : {results['fid_mean']:.4f} ± {results['fid_std']:.4f}")
    print(f"KID : {results['kid_mean']:.6f} ± {results['kid_std']:.6f}")
    print(f"{'='*50}")
    return results

# Execução base
BASE_CONFIG = {'latent_dim': 128, 'base_channels': 48, 'dropout': 0.05, 'epochs': 50, 'lr': 2e-4, 'beta': 0.7}
base_model = ConvVAE(**{k: BASE_CONFIG[k] for k in ['latent_dim', 'base_channels', 'dropout']}).to(device)
base_history, base_dir = train_vae(
    base_model,
    dev_loader,
    epochs=BASE_CONFIG['epochs'],
    lr=BASE_CONFIG['lr'],
    beta=BASE_CONFIG['beta'],
    run_name='vae_base_dev_artbench10',
)
plot_loss_curves(base_history, 'VAE base - dev subset')
show_vae_reconstructions(base_model, test_loader)

print("\nA recolher 5000 imagens reais...")
real_images = collect_real_images(test_loader, N_EVAL_SAMPLES)

print("\nA iniciar avaliação (10 repetições)...")
base_eval_results = evaluate_vae(base_model, real_images)

with open(base_dir / 'evaluation_base_model.json', 'w', encoding='utf-8') as f:
    json.dump(base_eval_results, f, indent=2)

## 3. Melhorias manuais no VAE

### 📝 Comparativo Técnico: VAE Base vs. Improved VAE

Nesta secção, documentamos a transição da arquitetura baseline para a versão otimizada, focada em resolver o problema do desfoque (blur) e aumentar a fidelidade das reconstruções para o dataset ArtBench-10.

| Ajuste Realizado | Razão Técnica | Impacto Esperado |
| :--- | :--- | :--- |
| **Capacidade (64 -> 512 filtros)** | Mais canais permitem à rede aprender texturas e pinceladas de alta frequência. | Menos perda de informação espacial e cores mais fiéis. |
| **Profundidade (4 camadas)** | Camada extra de processamento ajuda a extrair características mais abstratas. | Melhor compreensão da estrutura da imagem (objetos, formas). |
| **Espaço Latente (256-dim)** | Dobrar o tamanho do "gargalo" evita a "fome" de informação no decoder. | Reconstruções mais ricas e menos genéricas. |
| **Redução do Beta ($\beta = 0.4$)** | Foca o modelo mais na **reconstrução** do que na regularidade Gaussiana. | Ganho imediato de nitidez visual. |
| **Função de Perda (L1 / MAE)** | O L1 preserva melhor as arestas e variações de cor do que o MSE. | Definição superior nos contornos. |
| **Ajuste de LR (1e-4)** | Aumenta a estabilidade do treino numa rede mais profunda. | Convergência mais suave e sem "picos" de erro. |

---
**Observação:** A análise das curvas de treino demonstrou que o modelo **Improved** continuava a aprender mesmo após a época 60, ao contrário da versão Base que estagnou precocemente, validando o aumento de capacidade da rede.


In [ ]:
class ImprovedConvVAE(nn.Module):
    def __init__(self, latent_dim=256, base_channels=64, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        
        hidden_dim = base_channels * 8 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


IMPROVED_CONFIG = {
    'latent_dim': 256,
    'base_channels': 64,
    'dropout': 0.1,
    'epochs': 60,
    'lr': 1e-4,
    'beta': 0.4
}

improved_model = ImprovedConvVAE(
    latent_dim=IMPROVED_CONFIG['latent_dim'],
    base_channels=IMPROVED_CONFIG['base_channels'],
    dropout=IMPROVED_CONFIG['dropout'],
).to(device)

improved_history, improved_dir = train_vae(
    improved_model,
    dev_loader_aug,
    epochs=IMPROVED_CONFIG['epochs'],
    lr=IMPROVED_CONFIG['lr'],
    beta=IMPROVED_CONFIG['beta'],
    run_name='vae_improved_artbench10',
)

show_vae_reconstructions(improved_model, test_loader)
plot_loss_curves(improved_history, 'VAE Improved - dev subset')

# -----------------------------------------------------------------------------------------
# AVALIAÇÃO — protocolo do enunciado (FID + KID, 10 repetições)
# -----------------------------------------------------------------------------------------

print("\nA recolher 5000 imagens reais...")
real_images = collect_real_images(test_loader, N_EVAL_SAMPLES)

print("\nA iniciar avaliação do ImprovedConvVAE (10 repetições)...")
improved_eval_results = evaluate_vae(improved_model, real_images)

with open(improved_dir / 'evaluation_improved_model.json', 'w', encoding='utf-8') as f:
    json.dump(improved_eval_results, f, indent=2)

### 4 Otimização de Hiperparâmetros: Grid Search (Subset 20%)

Para identificar a arquitetura ideal e os melhores hiperparâmetros de treino antes de avançar para o dataset completo, implementámos uma *Grid Search* iterando sobre o subconjunto de 20% dos dados. Esta abordagem sistemática permite-nos analisar o compromisso entre a fidelidade visual, a diversidade e a estabilidade de treino.

#### 1. Espaço de Pesquisa (Search Space)

O nosso espaço de pesquisa foca-se em três eixos críticos para o desempenho do VAE, resultando num total de 8 combinações testadas ($2 \times 2 \times 2$):

| Parâmetro | Valores Testados | Justificação da Exploração |
| :--- | :--- | :--- |
| **Dimensão Latente** (`latent_dim`) | 256, 512 | Avaliar se um *bottleneck* mais largo previne a perda de detalhes finos das obras de arte sem comprometer a continuidade do espaço latente. |
| **Canais Base** (`base_channels`) | 64, 128 | Medir o impacto do aumento da capacidade de extração de características da rede (largura) na qualidade das texturas geradas. |
| **Termo de Regularização** ($\beta$) | 0.4, 0.6 | Afinar o equilíbrio crítico entre a minimização do erro de reconstrução e a restrição KL (*Kullback-Leibler*), minimizando o desfoque típico dos VAEs. |

**Parâmetros Fixos:**
* **Épocas:** 60
* **Learning Rate:** 1e-4
* **Métrica de Otimização:** O modelo ótimo é selecionado com base no menor valor médio de FID.

#### 2. Metodologia de Avaliação

Garantindo a robustez estatística exigida, cada uma das 8 configurações foi avaliada seguindo o protocolo estrito do projeto: geração de 5000 amostras comparadas com 5000 imagens reais, repetindo o processo de amostragem no mínimo 10 vezes com *seeds* aleatórias diferentes.


In [ ]:
import itertools

RUN_GRID_SEARCH = True
GRID_EPOCHS = 60

GRID_SEARCH_SPACE = {
    'latent_dim': [256, 512],
    'base_channels': [64, 128],
    'beta': [0.4, 0.6],
}

def run_grid_search(space, epochs=GRID_EPOCHS, real_images=None):
    keys = list(space.keys())
    results = []

    for values in itertools.product(*[space[k] for k in keys]):
        params = dict(zip(keys, values))
        lr = 1e-4
        run_name = f"lat{params['latent_dim']}_chan{params['base_channels']}_beta{params['beta']}"

        print(f"\n{'='*60}")
        print(f"A EXECUTAR RUN: {run_name} ({epochs} ÉP)")
        print(f"{'='*60}")

        model = ImprovedConvVAE(
            latent_dim=params['latent_dim'],
            base_channels=params['base_channels'],
        ).to(device)

        history, run_dir = train_vae(
            model,
            dev_loader_aug,
            epochs=epochs,
            lr=lr,
            beta=params['beta'],
            run_name='grid_' + run_name,
        )

        plot_loss_curves(history, f'Grid Search: {run_name}')
        show_vae_reconstructions(model, test_loader)

        # Avaliação FID/KID — protocolo do enunciado
        print(f"\nA avaliar {run_name}...")
        eval_results = evaluate_vae(model, real_images)

        with open(run_dir / 'evaluation.json', 'w', encoding='utf-8') as f:
            json.dump(eval_results, f, indent=2)

        results.append({
            'params': params,
            'recon_loss': history['recon_loss'][-1],
            'fid_mean': eval_results['fid_mean'],
            'fid_std':  eval_results['fid_std'],
            'kid_mean': eval_results['kid_mean'],
            'kid_std':  eval_results['kid_std'],
            'run_dir':  str(run_dir),
        })

        print(f"Done {run_name} | recon={results[-1]['recon_loss']:.4f} | FID={results[-1]['fid_mean']:.4f} ± {results[-1]['fid_std']:.4f} | KID={results[-1]['kid_mean']:.6f} ± {results[-1]['kid_std']:.6f}")

    # Ordenar por FID médio (métrica principal)
    return sorted(results, key=lambda x: x['fid_mean'])


if RUN_GRID_SEARCH:
    # Imagens reais recolhidas uma única vez para toda a grid search
    print("A recolher 5000 imagens reais...")
    real_images = collect_real_images(test_loader, N_EVAL_SAMPLES)

    grid_results = run_grid_search(GRID_SEARCH_SPACE, real_images=real_images)
    best = grid_results[0]

    print(f"\n{'='*60}")
    print(f"  MELHOR CONFIGURAÇÃO FINAL (por FID)")
    print(f"{'='*60}")
    print(f"  Parâmetros : {best['params']}")
    print(f"  Recon Loss : {best['recon_loss']:.4f}")
    print(f"  FID        : {best['fid_mean']:.4f} ± {best['fid_std']:.4f}")
    print(f"  KID        : {best['kid_mean']:.6f} ± {best['kid_std']:.6f}")
    print(f"{'='*60}")

    # Tabela resumo de todas as runs
    print(f"\n{'Run':<45} {'FID':>10} {'KID':>14}")
    print("-" * 72)
    for r in grid_results:
        name = f"lat{r['params']['latent_dim']}_chan{r['params']['base_channels']}_beta{r['params']['beta']}"
        print(f"{name:<45} {r['fid_mean']:>6.2f}±{r['fid_std']:<5.2f} {r['kid_mean']:>7.5f}±{r['kid_std']:<7.5f}")

else:
    print('Grid search desativada.')

| Modelo / Configuração | Latent Dim | Base Channels | Beta ($\beta$) | FID (Mean $\pm$ Std) | KID (Mean $\pm$ Std) |
| :--- | :---: | :---: | :---: | :--- | :--- |
| **VAE** | 512 | 128 | 0.6 | **373.26 $\pm$ 0.45** | **0.3708 $\pm$ 0.0020** |
| **VAE** | 512 | 128 | 0.4 | 382.13 $\pm$ 0.78 | 0.3914 $\pm$ 0.0016 |
| **VAE** | 512 | 64 | 0.6 | 396.92 $\pm$ 0.45 | 0.4061 $\pm$ 0.0014 |
| **VAE** | 512 | 64 | 0.4 | 407.94 $\pm$ 0.68 | 0.4319 $\pm$ 0.0022 |
| **VAE** | 256 | 128 | 0.6 | 413.08 $\pm$ 0.79 | 0.4195 $\pm$ 0.0024 |
| **VAE** | 256 | 128 | 0.4 | 388.84 $\pm$ 0.65 | 0.3867 $\pm$ 0.0024 |
| **VAE** | 256 | 64 | 0.6 | 397.73 $\pm$ 0.38 | 0.3979 $\pm$ 0.0016 |
| **VAE** | 256 | 64 | 0.4 | 410.30 $\pm$ 0.47 | 0.4273 $\pm$ 0.0018 |


###  5 .Treino Final e Monitorização de Métricas (Full Dataset)

Após a identificação da arquitetura e hiperparâmetros ótimos através da *Grid Search* (realizada no subconjunto de 20%), procedemos ao treino do modelo `ImprovedConvVAE` utilizando a totalidade do *dataset* de treino ArtBench-10.

#### 1. Configuração Final do Modelo

A configuração selecionada prioriza a capacidade de extração de características e a preservação de detalhes artísticos, utilizando os seguintes parâmetros:

| Hiperparâmetro | Valor | Descrição |
| :--- | :--- | :--- |
| **Dimensão Latente** | 512 | Espaço latente alargado para reter detalhes complexos das obras. |
| **Canais Base** | 128 | Maior largura da rede para processamento de texturas finas. |
| **Regularização ($\beta$)** | 0.6 | Fator de peso da divergência KL para equilibrar nitidez e continuidade estrutural. |
| **Taxa de Aprendizagem** | 1e-4 | Otimizador Adam com *learning rate* conservadora para estabilidade. |
| **Épocas** | 60 | Duração total do treino no *dataset* completo. |
| **Dropout** | 10% | Mitigação de *overfitting* nas camadas mais profundas do *encoder*. |

#### 2. Metodologia de Tracking Intermédio

Para compreender a dinâmica de aprendizagem do modelo e a evolução da qualidade gerativa ao longo do tempo, implementámos um sistema de *tracking* intermédio (*Fast Tracking Eval*):
* **Frequência:** A cada 5 épocas.
* **Amostragem:** 1000 amostras sintéticas contra 1000 imagens reais.
* **Objetivo:** Traçar as curvas de evolução do FID e KID sem incorrer no custo computacional proibitivo de executar o protocolo de avaliação completo a cada época. 
* **Considerações de Hardware:** Para garantir a compatibilidade e precisão dos cálculos do `TorchMetrics` (que requer suporte *float64*), o cálculo das métricas foi forçado para CPU em ambientes que utilizam a aceleração MPS (Apple Silicon).



In [ ]:
## 5. Treino final no conjunto completo com Tracking de Métricas

import copy
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
from tqdm import tqdm

# Importações de métricas (TorchMetrics)
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

# -------------------------------------------------------------------------
# A. CONFIGURAÇÃO DE HARDWARE 
# -------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# Métricas: CUDA suporta float64, MPS não — forçar CPU em MPS
if torch.backends.mps.is_available():
    METRICS_DEVICE = torch.device('cpu') 
else:
    METRICS_DEVICE = device

print(f"Treino a usar o device: {device}")
print(f"Métricas a usar o device: {METRICS_DEVICE}")

# -------------------------------------------------------------------------
# B. FUNÇÕES AUXILIARES E CONSTANTES
# -------------------------------------------------------------------------
N_EVAL_SAMPLES  = 5000
N_EVAL_REPEATS  = 10
KID_SUBSETS     = 50
KID_SUBSET_SIZE = 100
EVAL_BATCH_SIZE = 128

def vae_loss(recon, x, mu, logvar, beta=0.5):
    recon_loss = F.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_loss + beta * kl
    return total, recon_loss.detach(), kl.detach()

def collect_real_images(loader, n):
    collected, seen = [], 0
    for x, _, _ in loader:
        take = min(x.size(0), n - seen)
        collected.append(x[:take].cpu().float())
        seen += take
        if seen >= n:
            break
    return torch.cat(collected, dim=0)[:n]

@torch.no_grad()
def sample_vae_batched(model, n_samples, seed=None, batch_size=EVAL_BATCH_SIZE):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    
    out = []
    for i in range(0, n_samples, batch_size):
        out.append(model.decode(z[i:i+batch_size]).clamp(0.0, 1.0).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def sample_vae(model, n_samples, seed=None):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)

@torch.no_grad()
def evaluate_vae(model, real_images, repeats=N_EVAL_REPEATS, base_seed=0):
    fid_scores, kid_means, kid_stds = [], [], []

    for rep in range(repeats):
        seed = base_seed + rep
        print(f'  Rep {rep+1:02d}/{repeats}  (seed={seed})', end='  ', flush=True)

        fake_images = sample_vae_batched(model, N_EVAL_SAMPLES, seed=seed)

        fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
        kid_metric = KernelInceptionDistance(
            feature=2048, subsets=KID_SUBSETS, subset_size=KID_SUBSET_SIZE, normalize=False
        ).to(METRICS_DEVICE)

        for i in range(0, N_EVAL_SAMPLES, EVAL_BATCH_SIZE):
            real_batch = (real_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fake_batch = (fake_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            
            fid_metric.update(real_batch, real=True)
            fid_metric.update(fake_batch, real=False)
            
            kid_metric.update(real_batch, real=True)
            kid_metric.update(fake_batch, real=False)

        fid_val = float(fid_metric.compute().item())
        kid_mean, kid_std = kid_metric.compute()
        
        fid_scores.append(fid_val)
        kid_means.append(float(kid_mean.item()))
        kid_stds.append(float(kid_std.item()))

        print(f'FID = {fid_val:.4f} | KID = {kid_means[-1]:.6f} ± {kid_stds[-1]:.6f}')

        fid_metric.reset()
        kid_metric.reset()

    results = {
        'n_samples':        N_EVAL_SAMPLES,
        'repeats':          repeats,
        'fid_per_rep':      fid_scores,
        'kid_mean_per_rep': kid_means,
        'kid_std_per_rep':  kid_stds,
        'fid_mean':         float(np.mean(fid_scores)),
        'fid_std':          float(np.std(fid_scores)),
        'kid_mean':         float(np.mean(kid_means)),
        'kid_std':          float(np.std(kid_means)),
    }
    return results

# -------------------------------------------------------------------------
# C. DEFINIÇÃO DA REDE VAE MELHORADA
# -------------------------------------------------------------------------
class ImprovedConvVAE(nn.Module):
    def __init__(self, latent_dim=256, base_channels=64, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        
        hidden_dim = base_channels * 8 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


# -------------------------------------------------------------------------
# D. TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

# 1. Configuração Fixa Vencedora (Grid Search)
FINAL_CONFIG = {
    'latent_dim': 512,
    'base_channels': 128,
    'dropout': 0.1,
    'epochs': 60,
    'lr': 1e-4,
    'beta': 0.6
}

print(f"\nA iniciar treino final com a melhor configuração: {FINAL_CONFIG}")

final_model = ImprovedConvVAE(
    latent_dim=FINAL_CONFIG['latent_dim'],
    base_channels=FINAL_CONFIG['base_channels'],
    dropout=FINAL_CONFIG['dropout'],
).to(device)

# 2. Preparar Tracking (Acompanhamento rápido de FID/KID)
TRACKING_EPOCHS = 5 # Calcular métricas a cada 5 épocas para o gráfico
TRACKING_SAMPLES = 1000 # Menos amostras apenas para ter noção da tendência
tracking_history = {'epoch': [], 'fid': [], 'kid': []}

print("A recolher imagens reais para tracking durante o treino...")
real_tracking_images = collect_real_images(test_loader, TRACKING_SAMPLES)

def fast_tracking_eval(model, real_imgs):
    model.eval()
    fake_imgs = sample_vae_batched(model, TRACKING_SAMPLES, seed=42)
    
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
    kid_metric = KernelInceptionDistance(
        feature=2048, subsets=10, subset_size=50, normalize=False
    ).to(METRICS_DEVICE)
    
    for i in range(0, TRACKING_SAMPLES, EVAL_BATCH_SIZE):
        rb = (real_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fb = (fake_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fid_metric.update(rb, real=True)
        fid_metric.update(fb, real=False)
        kid_metric.update(rb, real=True)
        kid_metric.update(fb, real=False)
        
    fid_val = float(fid_metric.compute().item())
    kid_mean, _ = kid_metric.compute()
    return fid_val, float(kid_mean.item())

# 3. Treino Modificado com Tracking
# Se a constante OUTPUT_ROOT não estiver definida no teu script, ele vai gravar na pasta atual
from pathlib import Path
OUTPUT_ROOT = Path('./output') if 'OUTPUT_ROOT' not in locals() else OUTPUT_ROOT

run_dir = OUTPUT_ROOT / 'vae_final_full_artbench10'
run_dir.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.Adam(final_model.parameters(), lr=FINAL_CONFIG['lr'])
final_history = {'train_loss': [], 'recon_loss': [], 'kl_loss': []}

epochs = FINAL_CONFIG['epochs']
for epoch in range(1, epochs + 1):
    final_model.train()
    total_loss = total_recon = total_kl = 0.0
    progress = tqdm(full_train_loader_aug, desc=f'Epoch {epoch}/{epochs}', leave=False)
    
    for x, _, _ in progress:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = final_model(x)
        loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, beta=FINAL_CONFIG['beta'])
        loss.backward()
        optimizer.step()
        
        total_loss += float(loss.item())
        total_recon += float(recon_loss.item())
        total_kl += float(kl_loss.item())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    n_batches = max(1, len(full_train_loader_aug))
    final_history['train_loss'].append(total_loss / n_batches)
    final_history['recon_loss'].append(total_recon / n_batches)
    final_history['kl_loss'].append(total_kl / n_batches)
    
    print(f'Epoch {epoch:03d}/{epochs} | loss={final_history["train_loss"][-1]:.4f}')
    
    # Executar tracking do FID e KID
    if epoch % TRACKING_EPOCHS == 0 or epoch == epochs:
        f_val, k_val = fast_tracking_eval(final_model, real_tracking_images)
        tracking_history['epoch'].append(epoch)
        tracking_history['fid'].append(f_val)
        tracking_history['kid'].append(k_val)
        print(f" ---> Tracking Época {epoch}: FID={f_val:.2f} | KID={k_val:.4f}")

torch.save(final_model.state_dict(), run_dir / 'final_model.pt')

# 4. Desenhar os Gráficos Pedidos
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# (a) Total Loss Evolution
ax1.plot(range(1, epochs+1), final_history['train_loss'], color='skyblue', label='Total Loss')
ax1.set_title('(a) VAE Training Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.5)
ax1.legend()

# (b) FID Evolution
ax2.plot(tracking_history['epoch'], tracking_history['fid'], color='mediumpurple', marker='o', label='FID')
ax2.set_title('(b) FID Evolution')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('FID Score')
ax2.grid(True, alpha=0.5)
ax2.legend()

# (c) KID Evolution 
ax3.plot(tracking_history['epoch'], tracking_history['kid'], color='salmon', marker='o', label='KID')
ax3.set_title('(c) KID Evolution')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('KID Score')
ax3.grid(True, alpha=0.5)
ax3.legend()

plt.tight_layout()
plt.savefig(run_dir / 'training_evolution_charts.png')
plt.show()

# 5. Avaliação Final Oficial (Enunciado)
print("\nA recolher 5000 imagens reais para avaliação OFICIAL...")
real_images_final = collect_real_images(test_loader, N_EVAL_SAMPLES)

print("\nA iniciar avaliação final protocolar (10 repetições)...")
vae_eval = evaluate_vae(final_model, real_images_final)

with open(run_dir / 'evaluation_5000_samples_10_repeats.json', 'w', encoding='utf-8') as f:
    json.dump(vae_eval, f, indent=2)

print(f"\n{'='*50}")
print(f"RESULTADOS FINAIS — AVALIAÇÃO OFICIAL (10 REPs)")
print(f"{'='*50}")
print(f"FID : {vae_eval['fid_mean']:.4f} ± {vae_eval['fid_std']:.4f}")
print(f"KID : {vae_eval['kid_mean']:.6f} ± {vae_eval['kid_std']:.6f}")
print(f"{'='*50}")

# Grelha de amostras geradas
final_model.eval()
samples = sample_vae(final_model, 64, seed=2026).cpu()
save_image(make_grid(samples, nrow=8), run_dir / 'final_samples_grid.png')

# Mostrar grelha no ecrã
plt.figure(figsize=(8, 8))
plt.imshow(np.clip(make_grid(samples, nrow=8).permute(1, 2, 0).numpy(), 0, 1))
plt.axis('off')
plt.title('Amostras Geradas Finais')
plt.show()

print("\nDiretorio final com ficheiros e graficos:", run_dir)

### 6. Estudo Comparativo: Impacto das Funções de Perda na Qualidade Generativa

Conforme analisado na secção anterior, o desfoque (*blur*) característico das imagens geradas pelo VAE não é estritamente uma falha arquitetónica, mas sim uma consequência da otimização baseada na comparação direta de pixéis. Para validar empiricamente esta teoria e tentar extrair a máxima fidelidade visual da nossa arquitetura `ImprovedConvVAE`, desenhámos um estudo comparativo focado exclusivamente na função de erro de reconstrução.

Mantendo a arquitetura e os hiperparâmetros congelados (espaço latente de 512, 128 canais base, $\beta=0.6$), vamos submeter o modelo a três treinos distintos, alterando apenas a forma como o erro é calculado:

1. **Pixel-wise Loss (L1 - Baseline):** A nossa configuração padrão, baseada no Erro Médio Absoluto. Serve como ponto de controlo para demonstrar o comportamento conservador da rede, que tende a prever a "média" das cores em zonas de elevada complexidade (como pinceladas finas).
2. **Perceptual Loss (VGG-16):** Adoção de uma rede convolucional pré-treinada como extratora de características. O modelo será otimizado para minimizar a diferença nos mapas de *features* profundos (texturas, arestas e padrões complexos), em vez dos valores absolutos dos pixéis. Esta abordagem aproxima matematicamente o treino da forma como a métrica FID avalia as imagens.
3. **SSIM Loss (Structural Similarity Index):** Utilização de uma métrica de erro diferenciável que prioriza a preservação da estrutura espacial, covariância, luminância e contraste local das obras de arte, alinhando-se com a perceção visual humana.
4. **Super Loss (Abordagem Híbrida):** A culminação do nosso estudo exploratório. Esta configuração funde as três abordagens anteriores numa única função objetivo ponderada. Utiliza a métrica SSIM (84%) combinada com a L1 (16%) para estabelecer a estrutura base e a paleta de cores global de forma robusta, adicionando a *Perceptual Loss* (VGG-16) com um peso minoritário (5%) para forçar ativamente a geração de texturas de alta frequência e pinceladas, tudo sob a regularização latente (KL) característica do VAE.

**Objetivo da Experiência:**
Esta fase da nossa *pipeline* visa isolar o impacto da função objetivo. Ao avaliar os modelos sob o protocolo rigoroso do projeto (10 repetições de 5000 amostras), procuramos documentar quantitativamente (através do FID e KID) e qualitativamente (por inspeção visual) até que ponto a alteração da "régua matemática" de avaliação consegue superar as limitações inerentes da divergência KL e mitigar o desfoque do VAE. A expectativa é que a abordagem híbrida estabeleça o limite de fidelidade absoluto alcançável por um Autoencoder puro neste *dataset*.


## 6.2 Perceptual Loss (VGG-16)

In [ ]:
import copy
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from pathlib import Path

# Importações de métricas (TorchMetrics)
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image import StructuralSimilarityIndexMeasure

# -------------------------------------------------------------------------
# A. CONFIGURAÇÃO DE HARDWARE 
# -------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

if torch.backends.mps.is_available():
    METRICS_DEVICE = torch.device('cpu') 
else:
    METRICS_DEVICE = device

print(f"Treino a usar o device: {device}")
print(f"Métricas a usar o device: {METRICS_DEVICE}")

# -------------------------------------------------------------------------
# B. CONSTANTES E DEFINIÇÕES DE LOSS
# -------------------------------------------------------------------------
N_EVAL_SAMPLES  = 5000
N_EVAL_REPEATS  = 10
KID_SUBSETS     = 50
KID_SUBSET_SIZE = 100
EVAL_BATCH_SIZE = 128

# Instanciar SSIM
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# Classe VGG (com resize=False para poupar memória)
class VGGPerceptualLoss(nn.Module):
    def __init__(self, resize=False): 
        super().__init__()
        blocks = []
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        blocks.append(vgg[:4].eval())   
        blocks.append(vgg[4:9].eval())  
        blocks.append(vgg[9:16].eval()) 
        self.blocks = nn.ModuleList(blocks)
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.resize = resize

    def forward(self, input, target):
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std
        if self.resize:
            input = F.interpolate(input, mode='bilinear', size=(64, 64), align_corners=False)
            target = F.interpolate(target, mode='bilinear', size=(64, 64), align_corners=False)
        loss = 0.0
        x, y = input, target
        for block in self.blocks:
            x = block(x)
            y = block(y)
            loss += F.mse_loss(x, y)
        return loss

perceptual_criterion = VGGPerceptualLoss(resize=False).to(device)

# Funções de Perda Separadas
def vae_loss_vgg(recon, x, mu, logvar, beta=0.6, perceptual_weight=0.1):
    # L1 + VGG
    l1_loss = F.l1_loss(recon, x, reduction='mean')
    perc_loss = perceptual_criterion(recon, x)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = l1_loss + (perceptual_weight * perc_loss) + (beta * kl)
    return total, l1_loss.detach(), kl.detach(), perc_loss.detach()

def vae_loss_ssim(recon, x, mu, logvar, beta=0.6, ssim_alpha=0.84):
    # L1 + SSIM (Padrão Académico)
    l1_loss = F.l1_loss(recon, x, reduction='mean')
    ssim_val = ssim_metric(recon, x)
    ssim_loss = 1.0 - ssim_val
    recon_combined = (ssim_alpha * ssim_loss) + ((1.0 - ssim_alpha) * l1_loss)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_combined + (beta * kl)
    return total, recon_combined.detach(), kl.detach(), ssim_val.detach()

# (Restantes funções auxiliares: collect_real_images, sample_vae, evaluate_vae e a classe ImprovedConvVAE mantêm-se iguais e deves colá-las aqui)
def collect_real_images(loader, n):
    collected, seen = [], 0
    for x, _, _ in loader:
        take = min(x.size(0), n - seen)
        collected.append(x[:take].cpu().float())
        seen += take
        if seen >= n:
            break
    return torch.cat(collected, dim=0)[:n]

@torch.no_grad()
def sample_vae_batched(model, n_samples, seed=None, batch_size=EVAL_BATCH_SIZE):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    
    out = []
    for i in range(0, n_samples, batch_size):
        out.append(model.decode(z[i:i+batch_size]).clamp(0.0, 1.0).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def sample_vae(model, n_samples, seed=None):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)

@torch.no_grad()
def evaluate_vae(model, real_images, repeats=N_EVAL_REPEATS, base_seed=0):
    fid_scores, kid_means, kid_stds = [], [], []
    for rep in range(repeats):
        seed = base_seed + rep
        print(f'  Rep {rep+1:02d}/{repeats}  (seed={seed})', end='  ', flush=True)
        fake_images = sample_vae_batched(model, N_EVAL_SAMPLES, seed=seed)
        fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
        kid_metric = KernelInceptionDistance(feature=2048, subsets=KID_SUBSETS, subset_size=KID_SUBSET_SIZE, normalize=False).to(METRICS_DEVICE)
        for i in range(0, N_EVAL_SAMPLES, EVAL_BATCH_SIZE):
            real_batch = (real_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fake_batch = (fake_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fid_metric.update(real_batch, real=True)
            fid_metric.update(fake_batch, real=False)
            kid_metric.update(real_batch, real=True)
            kid_metric.update(fake_batch, real=False)
        fid_val = float(fid_metric.compute().item())
        kid_mean, kid_std = kid_metric.compute()
        fid_scores.append(fid_val)
        kid_means.append(float(kid_mean.item()))
        kid_stds.append(float(kid_std.item()))
        print(f'FID = {fid_val:.4f} | KID = {kid_means[-1]:.6f} ± {kid_stds[-1]:.6f}')
        fid_metric.reset()
        kid_metric.reset()
    results = {
        'fid_mean': float(np.mean(fid_scores)), 'fid_std': float(np.std(fid_scores)),
        'kid_mean': float(np.mean(kid_means)), 'kid_std': float(np.std(kid_means)),
    }
    return results

class ImprovedConvVAE(nn.Module):
    def __init__(self, latent_dim=512, base_channels=128, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1), nn.BatchNorm2d(base_channels * 2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1), nn.BatchNorm2d(base_channels * 4), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, 1, 1), nn.BatchNorm2d(base_channels * 8), nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        hidden_dim = base_channels * 8 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 3, 1, 1), nn.BatchNorm2d(base_channels * 4), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1), nn.BatchNorm2d(base_channels * 2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1), nn.Sigmoid()
        )
    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

# -------------------------------------------------------------------------
# D. TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

# -> ESCOLHE A TUA EXPERIÊNCIA AQUI <-
LOSS_TYPE = 'vgg'  # Opções: 'vgg' ou 'ssim'

FINAL_CONFIG = {
    'latent_dim': 512,
    'base_channels': 128,
    'dropout': 0.1,
    'epochs': 60,
    'lr': 1e-4,
    'beta': 0.6,
    'loss_type': LOSS_TYPE,
    'ssim_alpha': 0.84,        
    'perceptual_weight': 0.1  
}

print(f"\n[{LOSS_TYPE.upper()}] A iniciar treino com configuração: {FINAL_CONFIG}")

final_model = ImprovedConvVAE(
    latent_dim=FINAL_CONFIG['latent_dim'], base_channels=FINAL_CONFIG['base_channels'], dropout=FINAL_CONFIG['dropout']
).to(device)

TRACKING_EPOCHS = 5 
TRACKING_SAMPLES = 1000 
tracking_history = {'epoch': [], 'fid': [], 'kid': []}

print("A recolher imagens reais para tracking durante o treino...")
real_tracking_images = collect_real_images(test_loader, TRACKING_SAMPLES)

def fast_tracking_eval(model, real_imgs):
    model.eval()
    fake_imgs = sample_vae_batched(model, TRACKING_SAMPLES, seed=42)
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
    kid_metric = KernelInceptionDistance(feature=2048, subsets=10, subset_size=50, normalize=False).to(METRICS_DEVICE)
    for i in range(0, TRACKING_SAMPLES, EVAL_BATCH_SIZE):
        rb = (real_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fb = (fake_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fid_metric.update(rb, real=True); fid_metric.update(fb, real=False)
        kid_metric.update(rb, real=True); kid_metric.update(fb, real=False)
    fid_val = float(fid_metric.compute().item())
    kid_mean, _ = kid_metric.compute()
    return fid_val, float(kid_mean.item())

OUTPUT_ROOT = Path('./output') if 'OUTPUT_ROOT' not in locals() else OUTPUT_ROOT
run_dir = OUTPUT_ROOT / f'vae_final_{LOSS_TYPE}_artbench10'
run_dir.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.Adam(final_model.parameters(), lr=FINAL_CONFIG['lr'])

final_history = {'train_loss': [], 'specific_loss': [], 'kl_loss': []}
epochs = FINAL_CONFIG['epochs']

for epoch in range(1, epochs + 1):
    final_model.train()
    total_loss = total_specific = total_kl = 0.0
    progress = tqdm(full_train_loader_aug, desc=f'Epoch {epoch}/{epochs}', leave=False)
    
    for x, _, _ in progress:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = final_model(x)
        
        # Roteamento da Função de Perda
        if LOSS_TYPE == 'vgg':
            loss, _, kl, specific_val = vae_loss_vgg(recon, x, mu, logvar, beta=FINAL_CONFIG['beta'], perceptual_weight=FINAL_CONFIG['perceptual_weight'])
        elif LOSS_TYPE == 'ssim':
            loss, specific_val, kl, _ = vae_loss_ssim(recon, x, mu, logvar, beta=FINAL_CONFIG['beta'], ssim_alpha=FINAL_CONFIG['ssim_alpha'])
        
        loss.backward()
        optimizer.step()
        
        total_loss += float(loss.item())
        total_specific += float(specific_val.item())
        total_kl += float(kl.item())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    n_batches = max(1, len(full_train_loader_aug))
    final_history['train_loss'].append(total_loss / n_batches)
    final_history['specific_loss'].append(total_specific / n_batches)
    final_history['kl_loss'].append(total_kl / n_batches)
    
    print(f'Epoch {epoch:03d}/{epochs} | loss={final_history["train_loss"][-1]:.4f} (Específica [{LOSS_TYPE}]: {final_history["specific_loss"][-1]:.4f}, KL: {final_history["kl_loss"][-1]:.4f})')
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if epoch % TRACKING_EPOCHS == 0 or epoch == epochs:
        f_val, k_val = fast_tracking_eval(final_model, real_tracking_images)
        tracking_history['epoch'].append(epoch)
        tracking_history['fid'].append(f_val)
        tracking_history['kid'].append(k_val)
        print(f" ---> Tracking Época {epoch}: FID={f_val:.2f} | KID={k_val:.4f}")

torch.save(final_model.state_dict(), run_dir / 'final_model.pt')

# Gráficos
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
ax1.plot(range(1, epochs+1), final_history['train_loss'], color='skyblue', label='Total Loss')
ax1.set_title(f'(a) VAE Training Curve ({LOSS_TYPE.upper()})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=0.5); ax1.legend()

ax2.plot(tracking_history['epoch'], tracking_history['fid'], color='mediumpurple', marker='o', label='FID')
ax2.set_title('(b) FID Evolution')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('FID Score'); ax2.grid(True, alpha=0.5); ax2.legend()

ax3.plot(tracking_history['epoch'], tracking_history['kid'], color='salmon', marker='o', label='KID')
ax3.set_title('(c) KID Evolution')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('KID Score'); ax3.grid(True, alpha=0.5); ax3.legend()

plt.tight_layout()
plt.savefig(run_dir / 'training_evolution_charts.png')
plt.show()

# Avaliação Final
print("\nA iniciar avaliação final protocolar (10 repetições)...")
real_images_final = collect_real_images(test_loader, N_EVAL_SAMPLES)
vae_eval = evaluate_vae(final_model, real_images_final)

with open(run_dir / 'evaluation.json', 'w') as f:
    json.dump(vae_eval, f, indent=2)

print(f"\nRESULTADOS FINAIS [{LOSS_TYPE.upper()}] - FID: {vae_eval['fid_mean']:.4f} ± {vae_eval['fid_std']:.4f}")

final_model.eval()
samples = sample_vae(final_model, 64, seed=2026).cpu()
save_image(make_grid(samples, nrow=8), run_dir / 'final_samples_grid.png')

## 6.3 SSIM Loss (Structural Similarity Index)

In [ ]:
import copy
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from pathlib import Path

# Importações de métricas (TorchMetrics)
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image import StructuralSimilarityIndexMeasure

# -------------------------------------------------------------------------
# A. CONFIGURAÇÃO DE HARDWARE 
# -------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

if torch.backends.mps.is_available():
    METRICS_DEVICE = torch.device('cpu') 
else:
    METRICS_DEVICE = device

print(f"Treino a usar o device: {device}")
print(f"Métricas a usar o device: {METRICS_DEVICE}")

# -------------------------------------------------------------------------
# B. CONSTANTES E DEFINIÇÕES DE LOSS
# -------------------------------------------------------------------------
N_EVAL_SAMPLES  = 5000
N_EVAL_REPEATS  = 10
KID_SUBSETS     = 50
KID_SUBSET_SIZE = 100
EVAL_BATCH_SIZE = 128

# Instanciar SSIM
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# Classe VGG (com resize=False para poupar memória)
class VGGPerceptualLoss(nn.Module):
    def __init__(self, resize=False): 
        super().__init__()
        blocks = []
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        blocks.append(vgg[:4].eval())   
        blocks.append(vgg[4:9].eval())  
        blocks.append(vgg[9:16].eval()) 
        self.blocks = nn.ModuleList(blocks)
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.resize = resize

    def forward(self, input, target):
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std
        if self.resize:
            input = F.interpolate(input, mode='bilinear', size=(64, 64), align_corners=False)
            target = F.interpolate(target, mode='bilinear', size=(64, 64), align_corners=False)
        loss = 0.0
        x, y = input, target
        for block in self.blocks:
            x = block(x)
            y = block(y)
            loss += F.mse_loss(x, y)
        return loss

perceptual_criterion = VGGPerceptualLoss(resize=False).to(device)

# Funções de Perda Separadas
def vae_loss_vgg(recon, x, mu, logvar, beta=0.6, perceptual_weight=0.1):
    # L1 + VGG
    l1_loss = F.l1_loss(recon, x, reduction='mean')
    perc_loss = perceptual_criterion(recon, x)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = l1_loss + (perceptual_weight * perc_loss) + (beta * kl)
    return total, l1_loss.detach(), kl.detach(), perc_loss.detach()

def vae_loss_ssim(recon, x, mu, logvar, beta=0.6, ssim_alpha=0.84):
    # L1 + SSIM (Padrão Académico)
    l1_loss = F.l1_loss(recon, x, reduction='mean')
    ssim_val = ssim_metric(recon, x)
    ssim_loss = 1.0 - ssim_val
    recon_combined = (ssim_alpha * ssim_loss) + ((1.0 - ssim_alpha) * l1_loss)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon_combined + (beta * kl)
    return total, recon_combined.detach(), kl.detach(), ssim_val.detach()

# (Restantes funções auxiliares: collect_real_images, sample_vae, evaluate_vae e a classe ImprovedConvVAE mantêm-se iguais e deves colá-las aqui)
def collect_real_images(loader, n):
    collected, seen = [], 0
    for x, _, _ in loader:
        take = min(x.size(0), n - seen)
        collected.append(x[:take].cpu().float())
        seen += take
        if seen >= n:
            break
    return torch.cat(collected, dim=0)[:n]

@torch.no_grad()
def sample_vae_batched(model, n_samples, seed=None, batch_size=EVAL_BATCH_SIZE):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    
    out = []
    for i in range(0, n_samples, batch_size):
        out.append(model.decode(z[i:i+batch_size]).clamp(0.0, 1.0).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def sample_vae(model, n_samples, seed=None):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)

@torch.no_grad()
def evaluate_vae(model, real_images, repeats=N_EVAL_REPEATS, base_seed=0):
    fid_scores, kid_means, kid_stds = [], [], []
    for rep in range(repeats):
        seed = base_seed + rep
        print(f'  Rep {rep+1:02d}/{repeats}  (seed={seed})', end='  ', flush=True)
        fake_images = sample_vae_batched(model, N_EVAL_SAMPLES, seed=seed)
        fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
        kid_metric = KernelInceptionDistance(feature=2048, subsets=KID_SUBSETS, subset_size=KID_SUBSET_SIZE, normalize=False).to(METRICS_DEVICE)
        for i in range(0, N_EVAL_SAMPLES, EVAL_BATCH_SIZE):
            real_batch = (real_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fake_batch = (fake_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fid_metric.update(real_batch, real=True)
            fid_metric.update(fake_batch, real=False)
            kid_metric.update(real_batch, real=True)
            kid_metric.update(fake_batch, real=False)
        fid_val = float(fid_metric.compute().item())
        kid_mean, kid_std = kid_metric.compute()
        fid_scores.append(fid_val)
        kid_means.append(float(kid_mean.item()))
        kid_stds.append(float(kid_std.item()))
        print(f'FID = {fid_val:.4f} | KID = {kid_means[-1]:.6f} ± {kid_stds[-1]:.6f}')
        fid_metric.reset()
        kid_metric.reset()
    results = {
        'fid_mean': float(np.mean(fid_scores)), 'fid_std': float(np.std(fid_scores)),
        'kid_mean': float(np.mean(kid_means)), 'kid_std': float(np.std(kid_means)),
    }
    return results

class ImprovedConvVAE(nn.Module):
    def __init__(self, latent_dim=512, base_channels=128, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1), nn.BatchNorm2d(base_channels * 2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1), nn.BatchNorm2d(base_channels * 4), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, 1, 1), nn.BatchNorm2d(base_channels * 8), nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        hidden_dim = base_channels * 8 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 3, 1, 1), nn.BatchNorm2d(base_channels * 4), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1), nn.BatchNorm2d(base_channels * 2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1), nn.Sigmoid()
        )
    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

# -------------------------------------------------------------------------
# D. TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

# -> ESCOLHE A TUA EXPERIÊNCIA AQUI <-
LOSS_TYPE = 'ssim'  # Opções: 'vgg' ou 'ssim'

FINAL_CONFIG = {
    'latent_dim': 512,
    'base_channels': 128,
    'dropout': 0.1,
    'epochs': 60,
    'lr': 1e-4,
    'beta': 0.6,
    'loss_type': LOSS_TYPE,
    'ssim_alpha': 0.84,        
    'perceptual_weight': 0.1  
}

print(f"\n[{LOSS_TYPE.upper()}] A iniciar treino com configuração: {FINAL_CONFIG}")

final_model = ImprovedConvVAE(
    latent_dim=FINAL_CONFIG['latent_dim'], base_channels=FINAL_CONFIG['base_channels'], dropout=FINAL_CONFIG['dropout']
).to(device)

TRACKING_EPOCHS = 5 
TRACKING_SAMPLES = 1000 
tracking_history = {'epoch': [], 'fid': [], 'kid': []}

print("A recolher imagens reais para tracking durante o treino...")
real_tracking_images = collect_real_images(test_loader, TRACKING_SAMPLES)

def fast_tracking_eval(model, real_imgs):
    model.eval()
    fake_imgs = sample_vae_batched(model, TRACKING_SAMPLES, seed=42)
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
    kid_metric = KernelInceptionDistance(feature=2048, subsets=10, subset_size=50, normalize=False).to(METRICS_DEVICE)
    for i in range(0, TRACKING_SAMPLES, EVAL_BATCH_SIZE):
        rb = (real_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fb = (fake_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fid_metric.update(rb, real=True); fid_metric.update(fb, real=False)
        kid_metric.update(rb, real=True); kid_metric.update(fb, real=False)
    fid_val = float(fid_metric.compute().item())
    kid_mean, _ = kid_metric.compute()
    return fid_val, float(kid_mean.item())

OUTPUT_ROOT = Path('./output') if 'OUTPUT_ROOT' not in locals() else OUTPUT_ROOT
run_dir = OUTPUT_ROOT / f'vae_final_{LOSS_TYPE}_artbench10'
run_dir.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.Adam(final_model.parameters(), lr=FINAL_CONFIG['lr'])

final_history = {'train_loss': [], 'specific_loss': [], 'kl_loss': []}
epochs = FINAL_CONFIG['epochs']

for epoch in range(1, epochs + 1):
    final_model.train()
    total_loss = total_specific = total_kl = 0.0
    progress = tqdm(full_train_loader_aug, desc=f'Epoch {epoch}/{epochs}', leave=False)
    
    for x, _, _ in progress:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar = final_model(x)
        
        # Roteamento da Função de Perda
        if LOSS_TYPE == 'vgg':
            loss, _, kl, specific_val = vae_loss_vgg(recon, x, mu, logvar, beta=FINAL_CONFIG['beta'], perceptual_weight=FINAL_CONFIG['perceptual_weight'])
        elif LOSS_TYPE == 'ssim':
            loss, specific_val, kl, _ = vae_loss_ssim(recon, x, mu, logvar, beta=FINAL_CONFIG['beta'], ssim_alpha=FINAL_CONFIG['ssim_alpha'])
        
        loss.backward()
        optimizer.step()
        
        total_loss += float(loss.item())
        total_specific += float(specific_val.item())
        total_kl += float(kl.item())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    n_batches = max(1, len(full_train_loader_aug))
    final_history['train_loss'].append(total_loss / n_batches)
    final_history['specific_loss'].append(total_specific / n_batches)
    final_history['kl_loss'].append(total_kl / n_batches)
    
    print(f'Epoch {epoch:03d}/{epochs} | loss={final_history["train_loss"][-1]:.4f} (Específica [{LOSS_TYPE}]: {final_history["specific_loss"][-1]:.4f}, KL: {final_history["kl_loss"][-1]:.4f})')
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if epoch % TRACKING_EPOCHS == 0 or epoch == epochs:
        f_val, k_val = fast_tracking_eval(final_model, real_tracking_images)
        tracking_history['epoch'].append(epoch)
        tracking_history['fid'].append(f_val)
        tracking_history['kid'].append(k_val)
        print(f" ---> Tracking Época {epoch}: FID={f_val:.2f} | KID={k_val:.4f}")

torch.save(final_model.state_dict(), run_dir / 'final_model.pt')

# Gráficos
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
ax1.plot(range(1, epochs+1), final_history['train_loss'], color='skyblue', label='Total Loss')
ax1.set_title(f'(a) VAE Training Curve ({LOSS_TYPE.upper()})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=0.5); ax1.legend()

ax2.plot(tracking_history['epoch'], tracking_history['fid'], color='mediumpurple', marker='o', label='FID')
ax2.set_title('(b) FID Evolution')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('FID Score'); ax2.grid(True, alpha=0.5); ax2.legend()

ax3.plot(tracking_history['epoch'], tracking_history['kid'], color='salmon', marker='o', label='KID')
ax3.set_title('(c) KID Evolution')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('KID Score'); ax3.grid(True, alpha=0.5); ax3.legend()

plt.tight_layout()
plt.savefig(run_dir / 'training_evolution_charts.png')
plt.show()

# Avaliação Final
print("\nA iniciar avaliação final protocolar (10 repetições)...")
real_images_final = collect_real_images(test_loader, N_EVAL_SAMPLES)
vae_eval = evaluate_vae(final_model, real_images_final)

with open(run_dir / 'evaluation.json', 'w') as f:
    json.dump(vae_eval, f, indent=2)

print(f"\nRESULTADOS FINAIS [{LOSS_TYPE.upper()}] - FID: {vae_eval['fid_mean']:.4f} ± {vae_eval['fid_std']:.4f}")

final_model.eval()
samples = sample_vae(final_model, 64, seed=2026).cpu()
save_image(make_grid(samples, nrow=8), run_dir / 'final_samples_grid.png')

### 6.4 Super Loss (Abordagem Híbrida)


In [ ]:
import copy
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
from tqdm import tqdm

# Importações de métricas (TorchMetrics)
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image import StructuralSimilarityIndexMeasure

# -------------------------------------------------------------------------
# A. CONFIGURAÇÃO DE HARDWARE 
# -------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# Métricas: CUDA suporta float64, MPS não — forçar CPU em MPS
if torch.backends.mps.is_available():
    METRICS_DEVICE = torch.device('cpu') 
else:
    METRICS_DEVICE = device

print(f"Treino a usar o device: {device}")
print(f"Métricas a usar o device: {METRICS_DEVICE}")

# -------------------------------------------------------------------------
# B. FUNÇÕES AUXILIARES, LOSS E CONSTANTES
# -------------------------------------------------------------------------
N_EVAL_SAMPLES  = 5000
N_EVAL_REPEATS  = 10
KID_SUBSETS     = 50
KID_SUBSET_SIZE = 100
EVAL_BATCH_SIZE = 128

# 1. Instanciar SSIM
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# 2. Classe da Perceptual Loss (VGG16)
class VGGPerceptualLoss(nn.Module):
    # ATENÇÃO: resize=False para evitar o estouro de memória (OOM) que tiveste!
    def __init__(self, resize=False): 
        super().__init__()
        blocks = []
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        
        blocks.append(vgg[:4].eval())   # relu1_2
        blocks.append(vgg[4:9].eval())  # relu2_2
        blocks.append(vgg[9:16].eval()) # relu3_3
        
        self.blocks = nn.ModuleList(blocks)
        for param in self.parameters():
            param.requires_grad = False
            
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.resize = resize

    def forward(self, input, target):
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std
        
        if self.resize:
            input = F.interpolate(input, mode='bilinear', size=(64, 64), align_corners=False)
            target = F.interpolate(target, mode='bilinear', size=(64, 64), align_corners=False)
            
        loss = 0.0
        x, y = input, target
        for block in self.blocks:
            x = block(x)
            y = block(y)
            loss += F.mse_loss(x, y)
        return loss

perceptual_criterion = VGGPerceptualLoss(resize=False).to(device)

# 3. Super Loss Function: L1 + SSIM + VGG + KL
def vae_loss_combined(recon, x, mu, logvar, beta=0.6, perceptual_weight=0.05, ssim_alpha=0.84):
    # L1 Base
    l1_loss = F.l1_loss(recon, x, reduction='mean')
    
    # SSIM
    ssim_val = ssim_metric(recon, x)
    ssim_loss = 1.0 - ssim_val
    
    # Mistura clássica L1 e SSIM
    base_recon_loss = (ssim_alpha * ssim_loss) + ((1.0 - ssim_alpha) * l1_loss)
    
    # Perceptual VGG
    perc_loss = perceptual_criterion(recon, x)
    
    # KL Divergence
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Somatório final
    total = base_recon_loss + (perceptual_weight * perc_loss) + (beta * kl)
    
    return total, l1_loss.detach(), kl.detach(), perc_loss.detach(), ssim_val.detach()

def collect_real_images(loader, n):
    collected, seen = [], 0
    for x, _, _ in loader:
        take = min(x.size(0), n - seen)
        collected.append(x[:take].cpu().float())
        seen += take
        if seen >= n:
            break
    return torch.cat(collected, dim=0)[:n]

@torch.no_grad()
def sample_vae_batched(model, n_samples, seed=None, batch_size=EVAL_BATCH_SIZE):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    
    out = []
    for i in range(0, n_samples, batch_size):
        out.append(model.decode(z[i:i+batch_size]).clamp(0.0, 1.0).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def sample_vae(model, n_samples, seed=None):
    if seed is not None:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        z = torch.randn(n_samples, model.latent_dim, generator=g, device=device)
    else:
        z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)

@torch.no_grad()
def evaluate_vae(model, real_images, repeats=N_EVAL_REPEATS, base_seed=0):
    fid_scores, kid_means, kid_stds = [], [], []

    for rep in range(repeats):
        seed = base_seed + rep
        print(f'  Rep {rep+1:02d}/{repeats}  (seed={seed})', end='  ', flush=True)

        fake_images = sample_vae_batched(model, N_EVAL_SAMPLES, seed=seed)

        fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
        kid_metric = KernelInceptionDistance(
            feature=2048, subsets=KID_SUBSETS, subset_size=KID_SUBSET_SIZE, normalize=False
        ).to(METRICS_DEVICE)

        for i in range(0, N_EVAL_SAMPLES, EVAL_BATCH_SIZE):
            real_batch = (real_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            fake_batch = (fake_images[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
            
            fid_metric.update(real_batch, real=True)
            fid_metric.update(fake_batch, real=False)
            
            kid_metric.update(real_batch, real=True)
            kid_metric.update(fake_batch, real=False)

        fid_val = float(fid_metric.compute().item())
        kid_mean, kid_std = kid_metric.compute()
        
        fid_scores.append(fid_val)
        kid_means.append(float(kid_mean.item()))
        kid_stds.append(float(kid_std.item()))

        print(f'FID = {fid_val:.4f} | KID = {kid_means[-1]:.6f} ± {kid_stds[-1]:.6f}')

        fid_metric.reset()
        kid_metric.reset()

    results = {
        'n_samples':        N_EVAL_SAMPLES,
        'repeats':          repeats,
        'fid_per_rep':      fid_scores,
        'kid_mean_per_rep': kid_means,
        'kid_std_per_rep':  kid_stds,
        'fid_mean':         float(np.mean(fid_scores)),
        'fid_std':          float(np.std(fid_scores)),
        'kid_mean':         float(np.mean(kid_means)),
        'kid_std':          float(np.std(kid_means)),
    }
    return results

# -------------------------------------------------------------------------
# C. DEFINIÇÃO DA REDE VAE MELHORADA
# -------------------------------------------------------------------------
class ImprovedConvVAE(nn.Module):
    def __init__(self, latent_dim=512, base_channels=128, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 4, base_channels * 8, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 8),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        
        hidden_dim = base_channels * 8 * 4 * 4
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 3, 1, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

# -------------------------------------------------------------------------
# D. TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

FINAL_CONFIG = {
    'latent_dim': 512,
    'base_channels': 128,
    'dropout': 0.1,
    'epochs': 60,
    'lr': 1e-4,
    'beta': 0.6,
    'ssim_alpha': 0.84,        # Peso do SSIM vs L1
    'perceptual_weight': 0.05  # Peso da VGG um pouco mais baixo p/ equilibrar c/ SSIM
}

print(f"\nA iniciar treino final com SUPER LOSS (SSIM + VGG + L1 + KL): {FINAL_CONFIG}")

final_model = ImprovedConvVAE(
    latent_dim=FINAL_CONFIG['latent_dim'],
    base_channels=FINAL_CONFIG['base_channels'],
    dropout=FINAL_CONFIG['dropout'],
).to(device)

TRACKING_EPOCHS = 5 
TRACKING_SAMPLES = 1000 
tracking_history = {'epoch': [], 'fid': [], 'kid': []}

print("A recolher imagens reais para tracking durante o treino...")
real_tracking_images = collect_real_images(test_loader, TRACKING_SAMPLES)

def fast_tracking_eval(model, real_imgs):
    model.eval()
    fake_imgs = sample_vae_batched(model, TRACKING_SAMPLES, seed=42)
    
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(METRICS_DEVICE)
    kid_metric = KernelInceptionDistance(
        feature=2048, subsets=10, subset_size=50, normalize=False
    ).to(METRICS_DEVICE)
    
    for i in range(0, TRACKING_SAMPLES, EVAL_BATCH_SIZE):
        rb = (real_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fb = (fake_imgs[i:i+EVAL_BATCH_SIZE] * 255).byte().to(METRICS_DEVICE)
        fid_metric.update(rb, real=True)
        fid_metric.update(fb, real=False)
        kid_metric.update(rb, real=True)
        kid_metric.update(fb, real=False)
        
    fid_val = float(fid_metric.compute().item())
    kid_mean, _ = kid_metric.compute()
    return fid_val, float(kid_mean.item())

from pathlib import Path
OUTPUT_ROOT = Path('./output') if 'OUTPUT_ROOT' not in locals() else OUTPUT_ROOT

run_dir = OUTPUT_ROOT / 'vae_final_superloss_artbench10'
run_dir.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.Adam(final_model.parameters(), lr=FINAL_CONFIG['lr'])

# Registo abrangente
final_history = {'train_loss': [], 'l1_loss': [], 'kl_loss': [], 'perc_loss': [], 'ssim_metric': []}

epochs = FINAL_CONFIG['epochs']
for epoch in range(1, epochs + 1):
    final_model.train()
    total_loss = total_l1 = total_kl = total_perc = total_ssim = 0.0
    progress = tqdm(full_train_loader_aug, desc=f'Epoch {epoch}/{epochs}', leave=False)
    
    for x, _, _ in progress:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        recon, mu, logvar = final_model(x)
        
        loss, l1_loss, kl_loss, perc_loss, ssim_val = vae_loss_combined(
            recon, x, mu, logvar, 
            beta=FINAL_CONFIG['beta'], 
            perceptual_weight=FINAL_CONFIG['perceptual_weight'],
            ssim_alpha=FINAL_CONFIG['ssim_alpha']
        )
        
        loss.backward()
        optimizer.step()
        
        total_loss += float(loss.item())
        total_l1 += float(l1_loss.item())
        total_kl += float(kl_loss.item())
        total_perc += float(perc_loss.item())
        total_ssim += float(ssim_val.item())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    n_batches = max(1, len(full_train_loader_aug))
    final_history['train_loss'].append(total_loss / n_batches)
    final_history['l1_loss'].append(total_l1 / n_batches)
    final_history['kl_loss'].append(total_kl / n_batches)
    final_history['perc_loss'].append(total_perc / n_batches)
    final_history['ssim_metric'].append(total_ssim / n_batches)
    
    print(f'Epoch {epoch:03d}/{epochs} | loss={final_history["train_loss"][-1]:.4f} (SSIM: {final_history["ssim_metric"][-1]:.4f}, VGG: {final_history["perc_loss"][-1]:.4f}, KL: {final_history["kl_loss"][-1]:.4f})')
    
    # Prevenção OOM e memory leak
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Executar tracking do FID e KID
    if epoch % TRACKING_EPOCHS == 0 or epoch == epochs:
        f_val, k_val = fast_tracking_eval(final_model, real_tracking_images)
        tracking_history['epoch'].append(epoch)
        tracking_history['fid'].append(f_val)
        tracking_history['kid'].append(k_val)
        print(f" ---> Tracking Época {epoch}: FID={f_val:.2f} | KID={k_val:.4f}")

torch.save(final_model.state_dict(), run_dir / 'final_model.pt')

# 4. Desenhar os Gráficos Pedidos
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# (a) Total Loss Evolution
ax1.plot(range(1, epochs+1), final_history['train_loss'], color='skyblue', label='Total Loss')
ax1.set_title('(a) VAE Training Curve (Super Loss)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.5)
ax1.legend()

# (b) FID Evolution
ax2.plot(tracking_history['epoch'], tracking_history['fid'], color='mediumpurple', marker='o', label='FID')
ax2.set_title('(b) FID Evolution')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('FID Score')
ax2.grid(True, alpha=0.5)
ax2.legend()

# (c) KID Evolution 
ax3.plot(tracking_history['epoch'], tracking_history['kid'], color='salmon', marker='o', label='KID')
ax3.set_title('(c) KID Evolution')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('KID Score')
ax3.grid(True, alpha=0.5)
ax3.legend()

plt.tight_layout()
plt.savefig(run_dir / 'training_evolution_charts.png')
plt.show()

# 5. Avaliação Final Oficial
print("\nA recolher 5000 imagens reais para avaliação OFICIAL...")
real_images_final = collect_real_images(test_loader, N_EVAL_SAMPLES)

print("\nA iniciar avaliação final protocolar (10 repetições)...")
vae_eval = evaluate_vae(final_model, real_images_final)

with open(run_dir / 'evaluation_5000_samples_10_repeats.json', 'w', encoding='utf-8') as f:
    json.dump(vae_eval, f, indent=2)

print(f"\n{'='*50}")
print(f"RESULTADOS FINAIS — AVALIAÇÃO OFICIAL (10 REPs)")
print(f"{'='*50}")
print(f"FID : {vae_eval['fid_mean']:.4f} ± {vae_eval['fid_std']:.4f}")
print(f"KID : {vae_eval['kid_mean']:.6f} ± {vae_eval['kid_std']:.6f}")
print(f"{'='*50}")

# Grelha de amostras geradas
final_model.eval()
samples = sample_vae(final_model, 64, seed=2026).cpu()
save_image(make_grid(samples, nrow=8), run_dir / 'final_samples_grid.png')

# Mostrar grelha no ecrã
plt.figure(figsize=(8, 8))
plt.imshow(np.clip(make_grid(samples, nrow=8).permute(1, 2, 0).numpy(), 0, 1))
plt.axis('off')
plt.title('Amostras Geradas Finais (SSIM + VGG Loss)')
plt.show()

print("\nDiretorio final com ficheiros e graficos:", run_dir)

### Extra: Inadequação do Denoising Autoencoder (DAE) como Modelo Generativo

Durante a exploração arquitetónica inicial, implementámos um *Denoising Autoencoder* (DAE) padrão com o intuito de estabelecer um termo de comparação direto com o *Variational Autoencoder* (VAE). Como o objetivo central deste projeto incide sobre a geração de imagens e não sobre a classificação, é imperativo que o modelo selecionado seja capaz de produzir amostras inéditas e plausíveis que respeitem a distribuição dos dados originais.

Apesar de o DAE demonstrar elevada eficácia na aprendizagem de representações robustas — nomeadamente na eliminação de ruído e na reconstrução fiel da imagem de entrada —, o seu desempenho na vertente gerativa é manifestamente insuficiente devido à topologia do seu espaço latente. Esta limitação estrutural decorre de três fatores fundamentais:

* **Ausência de Regularização Latente:** Em contraste com o VAE, que recorre à divergência Kullback-Leibler (KL) para restringir as representações latentes a uma distribuição normal contínua $\mathcal{N}(0,I)$, o DAE não impõe qualquer limite matemático explícito à estruturação do seu espaço latente.
* **Topologia Descontínua:** Na falta de uma regularização prévia, as imagens de treino são projetadas em pontos isolados e dispersos de forma arbitrária no espaço multidimensional. Consequentemente, as vastas regiões situadas entre estes pontos mapeados permanecem "vazias", correspondendo a transições sem coerência semântica ou visual.
* **Inviabilidade de Amostragem Estocástica:** Ao tentar sintetizar uma nova imagem através da amostragem de um vetor de ruído aleatório (o procedimento padrão na geração estocástica), existe uma probabilidade elevadíssima de este vetor incidir numa dessas "zonas mortas" do espaço latente. Tal como evidenciado pelas nossas experiências exploratórias, o resultado visual traduz-se em artefactos distorcidos ou borrões ininteligíveis.

Em suma, o DAE opera estritamente como um mecanismo de compressão e reconstrução determinística. Sem o mapeamento contínuo proporcionado por uma distribuição prévia explícita, torna-se estruturalmente incapaz de atuar como um gerador fidedigno para a síntese de novas amostras artísticas no domínio do *dataset* ArtBench-10.

In [ ]:
## EXTRA: Denoising Autoencoder (DAE) - Demonstração de falha gerativa

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

class ConvDAE(nn.Module):
    def __init__(self, latent_dim=256, base_channels=64):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder determinístico (sem mu, sem logvar)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 4),
            nn.ReLU(inplace=True),
            nn.Flatten()
        )
        
        hidden_dim = base_channels * 4 * 4 * 4
        self.fc_encode = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, hidden_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_encode(h)

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(z.size(0), -1, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon

def train_dae(model, train_loader, epochs=10, lr=1e-4, noise_factor=0.2):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    print("A treinar o DAE (reconstrução com ruído)...")
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for x, _, _ in train_loader:
            x = x.to(device)
            
            # Adicionar ruído Gaussiano à imagem de entrada
            noise = torch.randn_like(x) * noise_factor
            noisy_x = torch.clamp(x + noise, 0., 1.)
            
            optimizer.zero_grad()
            recon = model(noisy_x)
            
            # A loss é apenas o erro de reconstrução (MSE) contra a imagem LIMPA
            loss = F.mse_loss(recon, x)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch}/{epochs} | Loss DAE: {total_loss/len(train_loader):.4f}")

# Inicializar e treinar rapidamente o DAE
dae_model = ConvDAE(latent_dim=256, base_channels=64).to(device)
train_dae(dae_model, dev_loader_aug, epochs=50)

# --- A PROVA DA FALHA GERATIVA ---

@torch.no_grad()
def sample_dae_random(model, n_samples):
    """Tenta gerar imagens a partir de ruído aleatório puro, como fazemos no VAE."""
    model.eval()
    # Amostrar do espaço Gaussiano normal N(0, 1)
    z = torch.randn(n_samples, model.latent_dim, device=device)
    return model.decode(z).clamp(0.0, 1.0)

# Gerar amostras a partir do nada
dae_samples = sample_dae_random(dae_model, 16)

plt.figure(figsize=(6, 6))
plt.imshow(np.clip(make_grid(dae_samples.cpu(), nrow=4).permute(1, 2, 0).numpy(), 0, 1))
plt.axis('off')
plt.title('Amostras Geradas pelo DAE (A partir de Ruído Aleatório)')
plt.show()

### Notas Finais: Análise Cruzada de Autoencoders (VAE vs. DAE)

A exploração da família de autoencoders no contexto do *dataset* ArtBench-10 permitiu isolar e compreender os mecanismos fundamentais necessários para a síntese de imagens.

Em estrito cumprimento com as diretrizes do projeto, o ***Variational Autoencoder* (VAE)** foi implementado e otimizado como a arquitetura representativa desta família. A sua formulação probabilística, que utiliza a divergência Kullback-Leibler (KL) para forçar o espaço latente a aproximar-se de uma distribuição normal contínua $\mathcal{N}(0,I)$, provou ser o elemento indispensável para viabilizar a amostragem estocástica. Contudo, apesar de garantir uma estabilidade de treino assinalável e de ter respondido positivamente à otimização de hiperparâmetros (como o aumento da capacidade da rede), o VAE confirmou a sua principal limitação empírica: a propensão para gerar imagens com desfoque (*blur*). 

Importa clarificar que esta perda de nitidez não constitui uma falha inultrapassável da topologia do modelo, mas sim um artefacto matemático da otimização baseada em erros *pixel a pixel* (como L1 ou *Mean Squared Error* - MSE). Perante a incerteza espacial na geração de texturas finas, o modelo minimiza o erro adotando um comportamento conservador, convergindo para o valor médio dos pixéis. Para trabalhos futuros, a fidelidade visual do VAE poderia ser substancialmente elevada substituindo ou complementando a métrica de reconstrução por funções de perda mais sofisticadas:

* **Perceptual Loss (VGG-16):** Em vez da comparação direta e isolada de pixéis, as imagens reais e geradas são processadas por uma rede convolucional pré-treinada. O erro é calculado comparando as representações profundas (*feature maps*), forçando o modelo a priorizar a coerência de texturas, arestas e estilos de alto nível, independentemente do alinhamento exato dos pixéis.
* **Structural Similarity Index (SSIM):** Adoção de uma métrica diferenciável focada na avaliação da degradação da estrutura espacial, luminância e contraste local. Ao contrário do MSE, a *loss* baseada em SSIM alinha-se de forma muito mais precisa com a perceção do sistema visual humano, mantendo a integridade das formas.
* **Super Loss Híbrida (SSIM + L1 + VGG):** A culminação do nosso estudo empírico sobre a função objetivo. Esta formulação multi-objetivo funde as abordagens anteriores: utiliza a SSIM (84%) combinada com a L1 (16%) para garantir a estrutura base e a exatidão da paleta de cores global, enquanto incorpora a *Perceptual Loss* para forçar ativamente a geração de texturas de alta frequência. Esta sinergia matemática superou drasticamente as limitações do desfoque, estabelecendo o teto de fidelidade visual do modelo.


Por outro lado, a implementação e análise do ***Denoising Autoencoder* (DAE)** — conduzidas como trabalho suplementar e investigativo — funcionaram como um excelente "controlo negativo" empírico. A avaliação do DAE demonstrou claramente que uma elevada capacidade de compressão e reconstrução determinística é insuficiente para a tarefa generativa. Na ausência da regularização contínua que caracteriza o VAE, o espaço latente do DAE permanece fragmentado e topologicamente desorganizado. Como resultado, qualquer tentativa de gerar imagens a partir de amostras de ruído aleatório traduz-se em artefactos visuais ininteligíveis.

Em suma, a contraposição destes dois modelos consolida a premissa de que a geração algorítmica de arte exige a modelação explícita de uma distribuição de probabilidade prévia. O VAE cumpre esse requisito de forma fiável, estabelecendo uma *baseline* robusta e estável contra a qual as arquiteturas de GANs (que resolvem o problema do desfoque via *adversarial loss*) e *Diffusion Models* poderão ser contrastadas em termos de fidelidade e diversidade visual.